## __Tópicos avanzados en Inteligencia Artificial 1 - MIA__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

**Asunto**: Generative Adversarial Networks (GANs)
*****

__Basada en la implementación:__ [Geeks for Geeks](https://www.geeksforgeeks.org/deep-learning/generative-adversarial-networks-gans-in-pytorch/)

## Librerias

In [ ]:
import matplotlib.pyplot as plt
import os

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms


## Setup CUDA

In [ ]:
## Verificar acceso de uso a CUDA

CUDA = False
seed = 0

CUDA = CUDA and torch.cuda.is_available()
print("PyTorch version: {}".format(torch.__version__))
if CUDA:
    print("CUDA version: {}\n".format(torch.version.cuda))

if CUDA:
    torch.cuda.manual_seed(seed)
device = torch.device("cuda:0" if CUDA else "cpu")

## Diseño del modelo

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_dim):
        """
            DESCRIPTION:
                Generator model builder

            INPUT:
                @param noise_dim: noise vector dimension
                @type noise_dim: int
        
        """
        super(Generator, self).__init__()
        self.noise_dim = noise_dim
        self.main = nn.Sequential(
            nn.Linear(noise_dim, 7 * 7 * 256),
            nn.ReLU(True),
            nn.Unflatten(1, (256, 7, 7)),
            nn.ConvTranspose2d(256, 128, 5, stride=1, padding=2),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 5, stride=2, padding=2, output_padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 1, 5, stride=2, padding=2, output_padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        return self.main(x)

class Discriminator(nn.Module):
    def __init__(self):
        """
            DESCRIPTION:
                Discriminator model builder
        """
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(1, 64, 5, stride=2, padding=2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 128, 5, stride=2, padding=2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm2d(128),
            nn.Flatten(),
            nn.Linear(7 * 7 * 128, 1)
        )

    def forward(self, x):
        return self.main(x)

def generate_and_save_images(model, epoch, noise, figsize=(12, 12)):
    """
        DESCRIPTION:
            Image generator from model

        INPUT:
            @param model: generator model
            @type model: torch.Model

            @param epoch: last epoch trained
            @type epoch: int

            @param noise: noise vector
            @type noise: torch.tensor

            @param figsize: figure space size (width, height) (default: (12, 12))
            @type figsize: tupla
    """

    ## Setting model to evaluate mode
    model.eval()
    with torch.no_grad():
        
        ## Generate images
        fake_images = model(noise).cpu()
        fake_images = fake_images.view(fake_images.size(0), 28, 28)

        fig = plt.figure(figsize=figsize)
        for i in range(fake_images.size(0)):
            plt.subplot(4, 4, i+1)
            plt.imshow(fake_images[i], cmap='gray')
            plt.axis('off')

        plt.savefig(f'image_at_epoch_{epoch+1:04d}.png')
        plt.show()


## Dataset

<center>
    <img src=https://upload.wikimedia.org/wikipedia/commons/f/f7/MnistExamplesModified.png width=800>
</center>

MNIST (Modified National Institute of Standards and Technology) es una gran base de datos de dígitos escritos a mano que se utiliza habitualmente para entrenar diversos sistemas de procesamiento de imágenes. La base de datos contiene 60.000 imágenes de entrenamiento

**Objetivo**: Clasificar las imágenes según su dígito.


#### Carga de datos

In [ ]:
## Parameters
NOISE_DIM = 100
NUM_EPOCHS = 50
BATCH_SIZE = 256

In [ ]:
## Descarga, instancia de gestor de datos y transformaciones
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

## Modelo

In [ ]:
## Instancias de los modelos
generator = Generator(NOISE_DIM).to(device)
discriminator = Discriminator().to(device)

## Funcion de perdida
criterion = nn.BCEWithLogitsLoss()

## Asigacion de optimizador para cada modelo
generator_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
discriminator_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

In [ ]:
## Entrenamiento de los modelos

for epoch in range(NUM_EPOCHS):
    for i, data in enumerate(train_loader):

        ## Cargar un batch de imágenes
        real_images, _ = data
        real_images = real_images.to(device)

        ## Entrenamiento el discriminador con datos reales
        discriminator_optimizer.zero_grad()
        real_labels = torch.ones(real_images.size(0), 1, device=device)
        real_outputs = discriminator(real_images)
        real_loss = criterion(real_outputs, real_labels)
        real_loss.backward()

        ## Entrenamiento el discriminador con datos generados
        noise = torch.randn(real_images.size(0), NOISE_DIM, device=device)
        fake_images = generator(noise)
        fake_labels = torch.zeros(real_images.size(0), 1, device=device)
        fake_outputs = discriminator(fake_images.detach())
        fake_loss = criterion(fake_outputs, fake_labels)
        fake_loss.backward()
        discriminator_optimizer.step()

        ## Entrenamiento del generador
        generator_optimizer.zero_grad()
        fake_labels = torch.ones(real_images.size(0), 1, device=device)
        fake_outputs = discriminator(fake_images)
        gen_loss = criterion(fake_outputs, fake_labels)
        gen_loss.backward()
        generator_optimizer.step()

        if i % 100 == 0:
            print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Step [{i+1}/{len(train_loader)}], '
                  f'Discriminator Loss: {real_loss.item() + fake_loss.item():.4f}, '
                  f'Generator Loss: {gen_loss.item():.4f}')

In [ ]:
## Guardar el modelo
#torch.save(discriminator, 'GAN_discriminator.pth')
#torch.save(generator, 'GAN_generator.pth')

Descarga de modelos entrenados.

* [Modelo generador](https://alumnosuaicl-my.sharepoint.com/:u:/g/personal/anthony_cho_l_edu_uai_cl/ETFkqJJG8sZMus92fa7KeVkBz8Vlirlqf5rB8q8fSmRbNA?e=TA8Upj)
* [Modelo discriminador](https://alumnosuaicl-my.sharepoint.com/:u:/g/personal/anthony_cho_l_edu_uai_cl/EXHNOWaz4xZDlzaOoK8sSlMBCVUFDmqUp2cdv7tffT0vgQ?e=Wg1qdu)

In [ ]:
if os.name == 'posix':

    ## Descaga del generador
    !wget "https://alumnosuaicl-my.sharepoint.com/:u:/g/personal/anthony_cho_l_edu_uai_cl/ETFkqJJG8sZMus92fa7KeVkBz8Vlirlqf5rB8q8fSmRbNA?e=TA8Upj&download=1"
    !mv "ETFkqJJG8sZMus92fa7KeVkBz8Vlirlqf5rB8q8fSmRbNA?e=TA8Upj&download=1" "GAN_generator.pth"

    ## Descarga del discriminador
    !wget "https://alumnosuaicl-my.sharepoint.com/:u:/g/personal/anthony_cho_l_edu_uai_cl/EXHNOWaz4xZDlzaOoK8sSlMBCVUFDmqUp2cdv7tffT0vgQ?e=Wg1qdu&download=1"
    !mv "EXHNOWaz4xZDlzaOoK8sSlMBCVUFDmqUp2cdv7tffT0vgQ?e=Wg1qdu&download=1" "GAN_discriminator.pth"


In [ ]:
## Cargar el modelo
#discriminator = torch.load('GAN_discriminator.pth', weights_only=False)
#generator = torch.load('GAN_generator.pth', weights_only=False)

#### Generar imagenes

In [ ]:
## Generar un vector de ruido
test_noise = torch.randn(16, NOISE_DIM, device=device)

## Generar imagenes a partir del vector de ruido
generate_and_save_images(generator, NUM_EPOCHS, test_noise)